# GeoAtt-PointNet++ v5.0.0 — Compare & Analyze

**Tujuan**: agregat hasil 10 seed × 2 varian → uji hipotesis statistik (paired Wilcoxon) untuk verdict Gate 2.

**Setup**:
- 10 seed: 42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4
- 2 varian: no_geom, with_geom
- Metrik: EER, AUC, TAR@FAR=1%, d-prime, Acc@EER
- Splits: Test (primary), Holdout (secondary generalization)

**Gate 2 verdict** (berdasarkan paired Wilcoxon Test EER):
- 🎉 p<0.05, with_geom menang → HIPOTESIS TERKONFIRMASI
- 🟡 p>0.10 → NEUTRAL → coba aux loss / dataset expansion
- 🔴 p<0.05, with_geom kalah → trigger F2.10 GAM architecture fix

## 1. Setup — GitHub Clone + Analysis in Repo

**Prerequisite**: simpan GitHub Personal Access Token sebagai Colab Secret:
1. Sidebar kiri → 🔑 Secrets → Add Secret
2. Name: `GITHUB_TOKEN`, Value: token dari https://github.com/settings/tokens
3. Toggle "Notebook access" ON

Notebook ini hanya membaca hasil training/eval dari repo — tidak perlu dataset raw.

In [ ]:
from google.colab import userdata
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- 1. Repo setup (no Drive needed) ----
# ---- 2. Clone repo + checkout colab branch ----
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/RZulfikri/Thesis.git'
REPO_DIR = Path('/content/Thesis')
PROJECT_ROOT = REPO_DIR / '3DCNN'
COLAB_BRANCH = 'colab'

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin

os.chdir(str(REPO_DIR))
ret = subprocess.run(['git', 'checkout', COLAB_BRANCH], capture_output=True)
if ret.returncode != 0:
    !git checkout -b {COLAB_BRANCH}
else:
    !git merge origin/main --no-edit --strategy-option theirs 2>/dev/null || true

!git config user.email "colab-runner@thesis.local"
!git config user.name "Colab Runner"

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(str(PROJECT_ROOT))

SEEDS = [42, 123, 2024, 7, 31337, 0, 1, 2, 3, 4]
VARIANTS = ["no_geom", "with_geom"]

# Results & output — di dalam repo (sudah ada dari training)
RUNS_DIR = PROJECT_ROOT / "runs" / "v5_lowdata"
EVAL_DIR = PROJECT_ROOT / "eval_results" / "v5_lowdata"

import datetime
TS = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
ANALYSIS_DIR = PROJECT_ROOT / "analysis" / f"v5_lowdata_{TS}"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print(f'EVAL_DIR  : {EVAL_DIR}')
print(f'RUNS_DIR  : {RUNS_DIR}')
print(f'ANALYSIS  : {ANALYSIS_DIR}')
print(f'Branch    : {COLAB_BRANCH}')


## Runtime Shutdown Guard

Proteksi compute: **atexit** auto-shutdown jika crash + explicit shutdown di cell terakhir.

In [ ]:
import atexit

_shutdown_triggered = False

def shutdown_colab(silent=False):
    """Shutdown Colab runtime untuk hemat compute."""
    global _shutdown_triggered
    if _shutdown_triggered:
        return
    _shutdown_triggered = True
    if not silent:
        print('\n' + '=' * 60)
        print('\U0001f50c Shutting down Colab runtime...')
        print('=' * 60)
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        if not silent:
            print(f'  Shutdown error: {e}')
            print('  Manual: Runtime > Disconnect and delete runtime')

# Register atexit — auto-fire jika kernel crash / unexpected exit
atexit.register(shutdown_colab, silent=True)
print('\u26a1 Runtime shutdown guard terdaftar (atexit)')
print('   \u2192 Auto-shutdown jika crash')
print('   \u2192 Explicit shutdown di cell terakhir setelah selesai')

## Git Save Helper

Fungsi untuk commit + push hasil ke branch `colab`.

In [ ]:
import subprocess, os

def git_save(message: str, push: bool = False):
    """Commit semua perubahan dan opsional push ke GitHub."""
    repo = str(REPO_DIR)
    os.chdir(repo)

    # Cari file > 95MB (GitHub limit 100MB) — skip dir yang belum ada
    find_dirs = [d for d in ['3DCNN/runs', '3DCNN/eval_results', '3DCNN/analysis']
                 if os.path.isdir(os.path.join(repo, d))]
    large = ''
    if find_dirs:
        try:
            large = subprocess.check_output(
                ['find'] + find_dirs + ['-size', '+95M', '-type', 'f'],
                text=True, stderr=subprocess.DEVNULL, cwd=repo
            ).strip()
        except subprocess.CalledProcessError:
            large = ''
    if large:
        print('\u26a0\ufe0f File > 95MB (skip dari git — file besar tidak di-track):')
        for f in large.split('\n'):
            print(f'  {f}')
            subprocess.run(['git', 'update-index', '--skip-worktree', f],
                           capture_output=True)

    subprocess.run(['git', 'add', '-A'], cwd=repo)
    result = subprocess.run(['git', 'diff', '--cached', '--quiet'], cwd=repo)
    if result.returncode == 0:
        print(f'  (nothing to commit)')
        return

    subprocess.run(['git', 'commit', '-m', message], cwd=repo)
    print(f'\u2705 Committed: {message}')

    if push:
        ret = subprocess.run(
            ['git', 'push', 'origin', COLAB_BRANCH],
            cwd=repo, capture_output=True, text=True
        )
        if ret.returncode == 0:
            print(f'\u2705 Pushed ke origin/{COLAB_BRANCH}')
        else:
            print(f'\u274c Push error: {ret.stderr}')
            print('Coba push manual dari cell terakhir')

    os.chdir(str(PROJECT_ROOT))

print('git_save() ready')


## 2. Load Results

In [ ]:
def load_results(eval_dir: Path, split: str = "test") -> pd.DataFrame:
    rows = []
    for variant in VARIANTS:
        for seed in SEEDS:
            f = eval_dir / variant / f"seed_{seed}" / split / "results.json"
            if not f.exists():
                print(f"  MISSING: {f}")
                continue
            with open(f) as fp:
                data = json.load(fp)
            for entry in data:
                row = {
                    'variant': variant,
                    'seed': seed,
                    'split': split,
                    'eer': entry.get('eer'),
                    'auc': entry.get('auc'),
                    'tar_at_far1': entry.get('tar_at_far1'),
                    'tar_at_far01': entry.get('tar_at_far01'),
                    'dprime': entry.get('dprime'),
                    'accuracy_at_eer': entry.get('accuracy_at_eer'),
                }
                # Identification metrics jika ada
                for k in ['rank1', 'rank5', 'rank10', 'mAP']:
                    if k in entry:
                        row[k] = entry[k]
                rows.append(row)
    return pd.DataFrame(rows)


df_test    = load_results(EVAL_DIR, split="test")
df_holdout = load_results(EVAL_DIR, split="holdout")

print(f'\nTest    : {len(df_test)} rows')
print(f'Holdout : {len(df_holdout)} rows')

df_test.head(20)

## 3. Aggregation Table (Mean ± Std)

In [ ]:
def aggregate(df: pd.DataFrame, metrics: list) -> pd.DataFrame:
    agg = df.groupby('variant')[metrics].agg(['mean', 'std', 'min', 'max'])
    return agg

METRICS = ['eer', 'auc', 'tar_at_far1', 'dprime', 'accuracy_at_eer']
# Add identification if present
for m in ['rank1', 'rank5', 'mAP']:
    if m in df_test.columns:
        METRICS.append(m)

print('=' * 70)
print('TEST SET — Mean ± Std (10 seeds)')
print('=' * 70)
agg_test = aggregate(df_test, METRICS)
print(agg_test.round(4))

print('\n' + '=' * 70)
print('HOLDOUT SET — Mean ± Std (10 seeds)')
print('=' * 70)
agg_holdout = aggregate(df_holdout, METRICS)
print(agg_holdout.round(4))

# Save CSV
agg_test.to_csv(ANALYSIS_DIR / 'aggregate_test.csv')
agg_holdout.to_csv(ANALYSIS_DIR / 'aggregate_holdout.csv')
print(f'\nSaved aggregate CSV ke {ANALYSIS_DIR}')

## 4. Paired Wilcoxon Test (Gate 2 Verdict)

In [ ]:
from scipy.stats import wilcoxon

def paired_test(df: pd.DataFrame, variant_a: str, variant_b: str, metric: str = 'eer'):
    """
    Paired Wilcoxon: variant_a vs variant_b pada metric, per seed.
    Asumsi: untuk setiap seed, kedua varian ada.
    """
    a = df[df.variant == variant_a].set_index('seed')[metric].sort_index()
    b = df[df.variant == variant_b].set_index('seed')[metric].sort_index()
    seeds_common = a.index.intersection(b.index)
    a = a.loc[seeds_common]
    b = b.loc[seeds_common]

    if len(a) < 3:
        return {'error': f'Not enough common seeds: {len(a)}'}

    diff = a - b  # positif = a > b
    try:
        stat, p = wilcoxon(a, b)
    except ValueError as e:
        stat, p = None, None

    return {
        'n_seeds': len(a),
        'metric': metric,
        f'{variant_a}_mean': float(a.mean()),
        f'{variant_a}_std':  float(a.std()),
        f'{variant_b}_mean': float(b.mean()),
        f'{variant_b}_std':  float(b.std()),
        'diff_mean': float(diff.mean()),
        'diff_std':  float(diff.std()),
        'wilcoxon_stat': float(stat) if stat is not None else None,
        'wilcoxon_p':    float(p)    if p    is not None else None,
    }

# Test EER (Gate 2 PRIMARY)
print('=' * 70)
print('GATE 2 PRIMARY — Test EER: with_geom vs no_geom (paired Wilcoxon)')
print('=' * 70)
res_test = paired_test(df_test, 'with_geom', 'no_geom', 'eer')
for k, v in res_test.items():
    print(f'  {k:25s} = {v}')

print('\nVerdict:')
p = res_test['wilcoxon_p']
diff = res_test['diff_mean']
if p is None:
    print('  Cannot compute (insufficient seeds)')
elif p < 0.05 and diff < 0:
    print(f'  HIPOTESIS TERKONFIRMASI (p={p:.4f}, with_geom EER lebih rendah by {-diff:.4f})')
elif p > 0.10:
    print(f'  NEUTRAL (p={p:.4f}, no significant difference)')
elif p < 0.05 and diff > 0:
    print(f'  HIPOTESIS BERMASALAH (p={p:.4f}, with_geom EER LEBIH TINGGI by {diff:.4f})')
    print('  -> Trigger F2.10 GAM architecture fix')
else:
    print(f'  Marginal (0.05 < p={p:.4f} < 0.10)')

# Holdout EER (Gate 2 SECONDARY)
print('\n' + '=' * 70)
print('GATE 2 SECONDARY — Holdout EER (generalization claim)')
print('=' * 70)
res_holdout = paired_test(df_holdout, 'with_geom', 'no_geom', 'eer')
for k, v in res_holdout.items():
    print(f'  {k:25s} = {v}')

# Save stats
with open(ANALYSIS_DIR / 'wilcoxon_tests.json', 'w') as f:
    json.dump({'test_eer': res_test, 'holdout_eer': res_holdout}, f, indent=2)
print(f'\nWilcoxon tests saved ke {ANALYSIS_DIR / "wilcoxon_tests.json"}')

## 5. Bootstrap CI untuk Δ EER

In [ ]:
def bootstrap_ci(a: np.ndarray, b: np.ndarray, n_resample: int = 1000, ci: float = 0.95, seed: int = 42):
    """
    Paired bootstrap CI untuk Δ = a - b.
    """
    rng = np.random.default_rng(seed)
    n = len(a)
    diffs = []
    for _ in range(n_resample):
        idx = rng.integers(0, n, n)
        diffs.append(np.mean(a[idx] - b[idx]))
    diffs = np.sort(diffs)
    lo = np.percentile(diffs, (1 - ci) / 2 * 100)
    hi = np.percentile(diffs, (1 + ci) / 2 * 100)
    return float(np.mean(diffs)), float(lo), float(hi)

for split_name, df in [('Test', df_test), ('Holdout', df_holdout)]:
    a = df[df.variant == 'with_geom'].set_index('seed')['eer'].sort_index().values
    b = df[df.variant == 'no_geom'].set_index('seed')['eer'].sort_index().values
    if len(a) == 0 or len(b) == 0:
        continue
    mean_diff, lo, hi = bootstrap_ci(a, b, n_resample=2000)
    print(f'{split_name:8s} Δ EER (with_geom - no_geom): {mean_diff:+.4f}  '
          f'95% CI: [{lo:+.4f}, {hi:+.4f}]  '
          f'{"(excludes 0)" if (lo > 0 or hi < 0) else "(includes 0)"}')

## 6. Boxplots — Test & Holdout

In [ ]:
metrics_to_plot = ['eer', 'auc', 'tar_at_far1', 'dprime']

fig, axes = plt.subplots(2, len(metrics_to_plot), figsize=(4*len(metrics_to_plot), 8))
for col_i, metric in enumerate(metrics_to_plot):
    # Test
    ax = axes[0, col_i]
    df_test.boxplot(column=metric, by='variant', ax=ax)
    ax.set_title(f'TEST: {metric}')
    ax.set_xlabel('')
    # Holdout
    ax = axes[1, col_i]
    df_holdout.boxplot(column=metric, by='variant', ax=ax)
    ax.set_title(f'HOLDOUT: {metric}')
    ax.set_xlabel('')

plt.suptitle('v5.0.0 Low-Data Regime — no_geom vs with_geom (10 seeds)', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'boxplots_test_holdout.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Per-Seed Comparison (Paired Differences)

In [ ]:
# Per-seed paired diff (with_geom - no_geom)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (split_name, df) in zip(axes, [('Test', df_test), ('Holdout', df_holdout)]):
    a = df[df.variant == 'with_geom'].set_index('seed')['eer'].sort_index()
    b = df[df.variant == 'no_geom'].set_index('seed')['eer'].sort_index()
    seeds_common = a.index.intersection(b.index)
    diff = (a.loc[seeds_common] - b.loc[seeds_common]).sort_index()

    colors = ['green' if d < 0 else 'red' for d in diff.values]
    ax.bar(range(len(diff)), diff.values, color=colors, alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xticks(range(len(diff)))
    ax.set_xticklabels([str(s) for s in diff.index], rotation=45)
    ax.set_ylabel('Δ EER (with_geom - no_geom)')
    ax.set_xlabel('Seed')
    ax.set_title(f'{split_name}: per-seed Δ EER\n(green=with_geom menang, red=no_geom menang)')
    ax.grid(True, alpha=0.3)

plt.suptitle('Per-Seed Paired Difference (Test EER & Holdout EER)', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'per_seed_paired_diff.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Aux Loss Trajectory (Diagnostic)

In [ ]:
# Cek apakah aux loss turun monoton (memastikan geom branch belajar)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, variant in zip(axes, VARIANTS):
    for seed in SEEDS:
        log_path = RUNS_DIR / variant / f'seed_{seed}' / 'train_log.csv'
        if not log_path.exists():
            continue
        try:
            df_log = pd.read_csv(log_path)
            if 'aux_loss' in df_log.columns and (df_log['aux_loss'] > 0).any():
                ax.plot(df_log['epoch'], df_log['aux_loss'], alpha=0.5, label=f'seed_{seed}')
        except Exception as e:
            print(f'  Error reading {log_path}: {e}')

    ax.set_title(f'{variant} — Aux Loss Trajectory')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Aux Loss')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'aux_loss_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Val EER Trajectory (Model Selection Sanity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, variant in zip(axes, VARIANTS):
    for seed in SEEDS:
        log_path = RUNS_DIR / variant / f'seed_{seed}' / 'train_log.csv'
        if not log_path.exists():
            continue
        try:
            df_log = pd.read_csv(log_path)
            if 'val_eer' in df_log.columns:
                vals = df_log['val_eer'].replace(-1, np.nan)
                ax.plot(df_log['epoch'], vals, alpha=0.5, label=f'seed_{seed}')
        except Exception as e:
            pass

    ax.set_title(f'{variant} — Val Pair EER Trajectory')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val Pair EER')
    ax.set_ylim(0, 0.5)
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'val_eer_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

## 9b. Training Loss Trajectory

In [ ]:
# Training loss (triplet + aux) per epoch
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, variant in zip(axes, VARIANTS):
    for seed in SEEDS:
        log_path = RUNS_DIR / variant / f'seed_{seed}' / 'train_log.csv'
        if not log_path.exists():
            continue
        try:
            df_log = pd.read_csv(log_path)
            if 'train_loss' in df_log.columns:
                ax.plot(df_log['epoch'], df_log['train_loss'],
                        alpha=0.5, label=f'seed_{seed}')
        except Exception:
            pass

    ax.set_title(f'{variant} \u2014 Training Loss Trajectory')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Training Loss (Triplet + Aux)')
    ax.legend(fontsize=8, ncol=2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'train_loss_trajectory.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. Export Final Tables

In [ ]:
# LaTeX tables untuk laporan thesis
def to_latex(agg: pd.DataFrame, caption: str) -> str:
    latex = agg.round(4).to_latex(
        caption=caption,
        label='tab:v5_' + caption.replace(' ', '_').lower(),
    )
    return latex

with open(ANALYSIS_DIR / 'table_test.tex', 'w') as f:
    f.write(to_latex(agg_test, 'v5.0.0 Low-Data Test Set'))

with open(ANALYSIS_DIR / 'table_holdout.tex', 'w') as f:
    f.write(to_latex(agg_holdout, 'v5.0.0 Low-Data Holdout Set'))

print(f'LaTeX tables saved ke {ANALYSIS_DIR}')

# Markdown summary untuk laporan
def _fmt(v):
    return f'{v:.4f}' if v is not None else 'N/A'

md_summary = f'''# v5.0.0 Low-Data Regime — Hasil Eksperimen

**Tanggal**: {TS}
**Setup**: 10 subjek × 15 sesi × 1 median frame = 150 frames
**Varian**: no_geom (PointNet++ murni) vs with_geom (GeoAtt full)
**Seeds**: {SEEDS}

## Test EER (Primary Metric)

{agg_test[['eer']].round(4).to_markdown()}

## Holdout EER (Generalization)

{agg_holdout[['eer']].round(4).to_markdown()}

## Statistical Test (Wilcoxon paired)

**Test EER**:
- with_geom mean: {res_test["with_geom_mean"]:.4f} ± {res_test["with_geom_std"]:.4f}
- no_geom   mean: {res_test["no_geom_mean"]:.4f} ± {res_test["no_geom_std"]:.4f}
- Δ (with-no): {res_test["diff_mean"]:+.4f}
- Wilcoxon stat: {res_test["wilcoxon_stat"]}, p: {_fmt(res_test["wilcoxon_p"])}

**Holdout EER**:
- with_geom mean: {res_holdout["with_geom_mean"]:.4f} ± {res_holdout["with_geom_std"]:.4f}
- no_geom   mean: {res_holdout["no_geom_mean"]:.4f} ± {res_holdout["no_geom_std"]:.4f}
- Δ (with-no): {res_holdout["diff_mean"]:+.4f}
- Wilcoxon stat: {res_holdout["wilcoxon_stat"]}, p: {_fmt(res_holdout["wilcoxon_p"])}
'''

with open(ANALYSIS_DIR / 'SUMMARY.md', 'w') as f:
    f.write(md_summary)

print(f'\nSummary markdown saved ke {ANALYSIS_DIR / "SUMMARY.md"}')
print('\nAll outputs:')
for f in sorted(ANALYSIS_DIR.iterdir()):
    print(f'  {f.name}')

## 11. Decision Branch (Gate 2 Outcome)

In [ ]:
# Berdasarkan hasil Wilcoxon, putuskan next step
p_test = res_test['wilcoxon_p']
diff_test = res_test['diff_mean']

print('=' * 70)
print('GATE 2 DECISION')
print('=' * 70)

if p_test is None:
    print('UNDEFINED — tidak cukup seed untuk Wilcoxon')
elif p_test < 0.05 and diff_test < 0:
    print(f'HIPOTESIS TERKONFIRMASI (p={p_test:.4f})')
    print('   Action: lanjut F2.8 replikasi all-frame untuk plot gap-vs-size')
    print('   Tag: v5.0.0-lowdata-confirmed')
elif p_test > 0.10:
    print(f'NEUTRAL (p={p_test:.4f}, no significant difference)')
    print('   Action: konsiderasi tambah seeds (15+) untuk lebih statistical power')
    print('   Atau: dokumentasikan sebagai "GeoAtt tidak terbukti merugikan"')
elif p_test < 0.05 and diff_test > 0:
    print(f'HIPOTESIS BERMASALAH (p={p_test:.4f})')
    print(f'   with_geom EER LEBIH TINGGI dari no_geom by {diff_test:.4f}')
    print('   Action: trigger F2.10 GAM architecture fix (FiLM modulation)')
    print('   Atau: F2.11 auxiliary loss weight tuning')
else:
    print(f'MARGINAL (0.05 < p={p_test:.4f} < 0.10)')
    print('   Action: tambah seeds atau accept verdict "trending favorable"')

print('\n' + '=' * 70)

## Git Push Results

Commit + push semua hasil ke branch `colab`.  
Di lokal: `git fetch origin && git checkout colab` untuk lihat hasil.

In [ ]:
# Final push — semua results ke GitHub branch 'colab'
git_save('v5.0.0 low-data: all results (training + eval + analysis)', push=True)

print('\n' + '=' * 60)
print('Untuk akses di lokal:')
print('  git fetch origin')
print('  git checkout colab')
print('  # atau: git pull origin colab')
print('=' * 60)


## Auto-Shutdown Compute

Otomatis shutdown runtime setelah semua proses selesai.  
Delay 60 detik untuk review output. Cancel: **Runtime > Interrupt execution**.

In [ ]:
import time

SHUTDOWN_DELAY = 60  # detik, beri waktu review hasil

print('=' * 60)
print('\u2705 ALL DONE — v5.0.0 Compare + Analysis selesai!')
print('=' * 60)
print(f'\nRuntime akan di-shutdown dalam {SHUTDOWN_DELAY} detik.')
print('Untuk membatalkan: Runtime > Interrupt execution\n')

try:
    for remaining in range(SHUTDOWN_DELAY, 0, -10):
        print(f'  Shutdown dalam {remaining} detik...')
        time.sleep(min(10, remaining))
    shutdown_colab()
except KeyboardInterrupt:
    print('\n\u26a0\ufe0f Shutdown dibatalkan. Runtime tetap aktif.')
    _shutdown_triggered = False  # reset supaya bisa dipanggil lagi